In [ ]:
# Fit the EE GPR contribution while keeping previously trained CC and EC contributions fixed

import time
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt


from pathlib import Path

DATA_DIR = Path("../data/training")

pairwise_dis_ce = [
    DATA_DIR / "edge_to_edge" / "PairDis_1_CC.npy",
    DATA_DIR / "face_to_face" / "PairDis_1_CC.npy",
]

pairwise_dis_ed_cen = [
    DATA_DIR / "edge_to_edge" / "PairDis_1_EC.npy",
    DATA_DIR / "face_to_face" / "PairDis_1_EC.npy",
]

pairwise_dis_cen_ed = [
    DATA_DIR / "edge_to_edge" / "PairDis_1_CE.npy",
    DATA_DIR / "face_to_face" / "PairDis_1_CE.npy",
]

pairwise_dis_ed = [
    DATA_DIR / "edge_to_edge" / "PairDis_1_EE.npy",
    DATA_DIR / "face_to_face" / "PairDis_1_EE.npy",
]

Actual_data = [
    DATA_DIR / "edge_to_edge" / "distance_edge_1_ext.out",
    DATA_DIR / "face_to_face" / "dspacing_PMF.out",
]


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

#Optimized parameters from previous stages
morse_cc = torch.tensor([-200.3445,    0.8255,   -2.2312], device=device)
num_S = 38
S_points = torch.tensor([0.0000,  1.0417,  2.0833,  2.6000,  3.1250,  3.6000,  4.1667,  4.6000,
         5.2083,  6.2500,  6.6000,  7.2917,  7.6000,  8.3333,  8.4000,  9.3750,
         9.6000, 10.4000, 10.4167, 11.4583, 11.6000, 12.4000, 12.5000, 13.5417,
        13.6000, 14.5833, 14.6000, 15.6000, 15.6250, 16.6667, 17.7083, 18.7500,
        19.7917, 20.8333, 21.8750, 22.9167, 23.9583, 25.0000
], device=device)


l_cc = torch.tensor(0.7842, device=device)

coeffs_cc = torch.tensor([0.0000e+00,  0.0000e+00,  6.3404e-07,  1.8891e+00,  8.1625e+00,
         1.0581e-01, -2.4098e+00, -4.0319e-01,  7.9287e-01, -1.9698e-01,
        -6.0995e-02,  7.0925e-02, -4.9648e-02, -5.2572e-02,  4.9121e-02,
         5.2969e-02, -2.1126e-02,  2.1917e-02, -5.2105e-02,  3.6958e-02,
        -4.6758e-02,  5.6710e-02, -1.1014e-02, -3.1777e-03, -1.9909e-02,
         6.7422e-03,  1.3268e-02, -1.1562e-03, -6.4095e-03,  4.7646e-03,
        -3.3277e-03,  2.0708e-03, -1.1522e-03,  6.7129e-04, -4.4138e-04,
         3.0538e-04, -1.4848e-04,  3.1906e-05
], device=device)

morse_ec = torch.tensor([-60.6008,   0.8054,  -0.4834], device=device)

l_ec = torch.tensor(0.9172, device=device)
coeffs_ec = torch.tensor([1.0000e-02,  1.0000e-02,  2.7176e-01,  1.8848e+00,  2.6622e+00,
        -8.0390e-01,  2.4433e+00,  5.5400e-02, -3.3196e+00,  2.7279e+00,
         4.3200e-02, -1.8347e+00, -2.9200e-02,  1.0011e+00,  2.5900e-02,
        -4.3774e-01, -5.2200e-02,  3.0100e-02,  1.9532e-01, -1.3795e-01,
        -6.3000e-03, -1.1700e-02,  1.2568e-01, -1.1154e-01, -7.1000e-03,
         9.7802e-02,  0.0000e+00,  0.0000e+00, -7.8401e-02,  6.1313e-02,
        -4.4903e-02,  3.1545e-02, -2.1325e-02,  1.4045e-02, -8.1299e-03,
         3.8430e-03, -9.9988e-04, -2.5886e-04
], device=device)


morse_ee = torch.tensor([-270.0342,    1.0854,   -1.2065], device = device)

def morse_potential(r, D_e, a, r_e):
    delta = a * (r - r_e)
    exp_term = torch.exp(-delta)
    return D_e * (1 - exp_term)**2 - D_e

def calculate_kernels(distances, S_points, l):
    l = torch.clamp(l, min=1e-3)  # Ensure l > 0

    # Compute squared differences for all pairs (distances, S_points)
    diff = (distances.unsqueeze(1) - S_points) ** 2 / (2 * (l ** 2))  # Ensure 3D shape

    # Compute Gaussian kernel
    kernel_values = torch.exp(-diff)

    # Compute cutoff value at r = 3l
    cutoff_value = torch.exp(-((3 * l) ** 2) / (2 * (l ** 2)))

    # Apply cutoff: subtract cutoff_value
    kernel_values = kernel_values.clone() - cutoff_value  

    # Apply masking for cutoff
    mask = torch.abs(distances.unsqueeze(1) - S_points) > 3 * l
    kernel_values = torch.where(mask, torch.tensor(0.0, device=l.device), kernel_values)

    return kernel_values  # Ensures (N, S) shape


def precompute_cc_contributions(dist, morse, S_points, l, coeffs):
    morse_list = []
    pred_list = []

    for dist_tensor in dist:
        # Morse potential energy
        morse_energy = morse_potential(dist_tensor, *morse).sum()
        morse_list.append(morse_energy)

        # GPR contribution
        K = calculate_kernels(dist_tensor, S_points, l)
        pred = torch.einsum('ns,s->n', K, coeffs).sum()
        pred_list.append(pred)

    # Convert lists to tensors
    morse_tensor = torch.stack(morse_list)
    pred_tensor = torch.stack(pred_list)

    return morse_tensor, pred_tensor

def precompute_ee_morse(dist_ee, morse_ee):
    morse_ee_list = []

    for dist_tensor in dist_ee:

        morse_energy = morse_potential(dist_tensor, *morse_ee).sum()
        morse_ee_list.append(morse_energy)


    morse_ee_tensor = torch.stack(morse_ee_list)

    return morse_ee_tensor


def morse_potential(r, D_e, a, r_e):
    delta = a * (r - r_e)
    exp_term = torch.exp(-delta)
    return D_e * (1 - exp_term)**2 - D_e

def calculate_kernels(distances, S_points, l):
    l = torch.clamp(l, min=1e-3)  # Ensure l > 0

    # Compute squared differences for all pairs (distances, S_points)
    diff = (distances.unsqueeze(1) - S_points) ** 2 / (2 * (l ** 2))  # Ensure 3D shape

    # Compute Gaussian kernel
    kernel_values = torch.exp(-diff)

    # Compute cutoff value at r = 3l
    cutoff_value = torch.exp(-((3 * l) ** 2) / (2 * (l ** 2)))

    # Apply cutoff: subtract cutoff_value
    kernel_values = kernel_values.clone() - cutoff_value  

    # Apply masking for cutoff
    mask = torch.abs(distances.unsqueeze(1) - S_points) > 3 * l
    kernel_values = torch.where(mask, torch.tensor(0.0, device=l.device), kernel_values)

    return kernel_values  # Ensures (N, S) shape


def precompute_cc_contributions(dist_cc, morse_cc, S_points, l_cc, coeffs_cc):
    morse_cc_list = []
    pred_cc_list = []

    for dist_tensor in dist_cc:
        # Morse potential energy
        morse_energy = morse_potential(dist_tensor, *morse_cc).sum()
        morse_cc_list.append(morse_energy)

        # GPR contribution
        K_cc = calculate_kernels(dist_tensor, S_points, l_cc)
        pred_cc = torch.einsum('ns,s->n', K_cc, coeffs_cc).sum()
        pred_cc_list.append(pred_cc)

    # Convert lists to tensors
    morse_cc_tensor = torch.stack(morse_cc_list)
    pred_cc_tensor = torch.stack(pred_cc_list)

    return morse_cc_tensor, pred_cc_tensor

def precompute_ec_morse(dist_ec, morse_ec):
    morse_ec_list = []

    for dist_tensor in dist_ec:

        morse_energy = morse_potential(dist_tensor, *morse_ec).sum()
        morse_ec_list.append(morse_energy)


    morse_ec_tensor = torch.stack(morse_ec_list)

    return morse_ec_tensor


def load_pairwise_distances(file_path):
    data = np.load(file_path, allow_pickle=True).item()  # Load as dictionary
    if not isinstance(data, dict):
        raise ValueError(f"Loaded data is not a dictionary: {type(data)} from {file_path}")


    sorted_keys = sorted(data.keys())
    d_spacing = torch.tensor(sorted_keys, dtype=torch.float32, device=device)
    distances = [torch.tensor(data[key], dtype=torch.float32, device=device) for key in sorted_keys]

    return d_spacing, distances

# Load actual energies and pairwise distances
processed_data = {}
actual_energies = {}

for i, file_path in enumerate(pairwise_dis_ce):
    dataset_key = f'dataset_{i+1}'
    processed_data[dataset_key] = {}

    # Load pairwise distances
    d_spacing_cc, dist_cc = load_pairwise_distances(file_path)
    processed_data[dataset_key]["d_spacing"] = d_spacing_cc
    processed_data[dataset_key]["center_center"] = dist_cc  # List of tensors
    

    print(f"✔ Loaded {file_path} - {len(dist_cc)} d-spacing values.")
    
for i, file_path in enumerate(pairwise_dis_ed_cen):
    dataset_key = f'dataset_{i+1}'
#     processed_data[dataset_key] = {}

    # Load pairwise distances
    d_spacing_ec, dist_ec = load_pairwise_distances(file_path)
    processed_data[dataset_key]["d_spacing"] = d_spacing_ec
    processed_data[dataset_key]["edge_center"] = dist_ec  # List of tensors
    

    print(f"✔ Loaded {file_path} - {len(dist_ec)} d-spacing values.")
    
    
for i, file_path in enumerate(pairwise_dis_cen_ed):
    dataset_key = f'dataset_{i+1}'
#     processed_data[dataset_key] = {}

    # Load pairwise distances
    d_spacing_ce, dist_ce = load_pairwise_distances(file_path)
    processed_data[dataset_key]["d_spacing"] = d_spacing_ce
    processed_data[dataset_key]["center_edge"] = dist_ce  # List of tensors
    

    print(f"✔ Loaded {file_path} - {len(dist_ce)} d-spacing values.")
    
    
    
for i, file_path in enumerate(pairwise_dis_ed):
    dataset_key = f'dataset_{i+1}'
#     processed_data[dataset_key] = {}

    # Load pairwise distances
    d_spacing_ee, dist_ee = load_pairwise_distances(file_path)
    processed_data[dataset_key]["d_spacing"] = d_spacing_ee
    processed_data[dataset_key]["edge_edge"] = dist_ee  # List of tensors
    

    print(f"✔ Loaded {file_path} - {len(dist_ee)} d-spacing values.")

# Load actual energy data
for i, file_path in enumerate(Actual_data):
    dataset_key = f'dataset_{i+1}'
    data = np.loadtxt(file_path, delimiter=' ')

    # Sort data by d-spacing
    sorted_indices = np.argsort(data[:, 0])
    data = data[sorted_indices]  # Apply sorting

    d_space = torch.tensor(data[:, 0], dtype=torch.float32, device=device)
    energy_values = torch.tensor(data[:, 1], dtype=torch.float32, device=device)

    # Store in dictionaries
    processed_data[dataset_key]["d_spacing_energy"] = d_space
    processed_data[dataset_key]["actual"] = energy_values
    actual_energies[dataset_key] = energy_values

    # Check if d-spacing values match between .npy and .out files
    if not torch.allclose(d_space, processed_data[dataset_key]["d_spacing"]):
        print(f"⚠ Warning: d-spacing mismatch between .npy and .out files for {dataset_key}")

    print(f"✔ Loaded {file_path} - {len(data)} rows sorted.")

for dkey in processed_data.keys():
    # For each d-spacing, concatenate edge_center and center_edge tensors
    processed_data[dkey]["mix"] = [
        torch.cat([ec, ce]) 
        for ec, ce in zip(
            processed_data[dkey]["edge_center"], 
            processed_data[dkey]["center_edge"]
        )
    ]
    
    
# Verify loaded data
for key in processed_data:
    print(f"\nDataset: {key}")
    print(f"  d_spacing (from .npy): {processed_data[key]['d_spacing'].shape}")
    print(f"  d_spacing (from .out): {processed_data[key]['d_spacing_energy'].shape}")
    print(f"  center_center distances: {len(processed_data[key]['center_center'])} arrays")
    print(f"  edge_center distances: {len(processed_data[key]['edge_center'])} arrays")
    print(f"  center_edge distances: {len(processed_data[key]['center_edge'])} arrays")
    print(f"  mix distances: {len(processed_data[key]['mix'])} arrays")
    
    print(f"  edge_edge distances: {len(processed_data[key]['edge_edge'])} arrays")
    
    
    print(f"  Actual energies: {processed_data[key]['actual'].shape}")

    # Print first 5 d-spacing values and corresponding distances/energies
    print("\nFirst 5 d-spacing values and corresponding data:")
    for i in range(5):
        print(f"  d-spacing: {processed_data[key]['d_spacing'][i].item()}")
        print(f"    Number of distances_cc: {len(processed_data[key]['center_center'][i])}")
        print(f"    Number of distances_ec: {len(processed_data[key]['edge_center'][i])}")
        print(f"    Number of distances_ec: {len(processed_data[key]['center_edge'][i])}")
        print(f"    Number of distances_mix: {len(processed_data[key]['mix'][i])}")
        
        print(f"    Number of distances_ee: {len(processed_data[key]['edge_edge'][i])}")
        
        print(f"    Energy: {processed_data[key]['actual'][i].item()}")



        
precomputed = {}
for dkey in processed_data:
    dist_cc = processed_data[dkey]["center_center"]
    dist_mix = processed_data[dkey]["mix"]
    dist_ee = processed_data[dkey]["edge_edge"]
    actual = processed_data[dkey]["actual"] 
    d_spacing =  processed_data[dkey]["d_spacing"] 
    
    
    morse_cc_tensor, pred_cc_tensor = precompute_cc_contributions(dist_cc, morse_cc, S_points, l_cc, coeffs_cc)
    morse_ec_tensor, pred_ec_tensor = precompute_cc_contributions(dist_mix, morse_ec, S_points, l_ec, coeffs_ec)
    
    morse_ee_tensor = precompute_ee_morse(dist_ee, morse_ee)
    
    precomputed[dkey] = {
        "morse_cc": morse_cc_tensor,
        "pred_cc": pred_cc_tensor,
        "morse_ec": morse_ec_tensor,
        "pred_ec": pred_ec_tensor,
        "morse_ee": morse_ee_tensor,
        "actual": actual,
        "dist_ee": dist_ee ,
        "dist_mix": dist_mix ,
        "dist_cc": dist_cc ,
        "d_spacing": d_spacing
        
    }                




In [ ]:
# Final EE GPR parameters used in the model
class CombinedGPRModel(nn.Module):
    def __init__(self, num_S=38):
        super().__init__()
        self.num_S = num_S
        self.S_points = torch.tensor([0.0000,  1.0417,  2.0833,  2.6000,  3.1250,  3.6000,  4.1667,  4.6000,
                 5.2083,  6.2500,  6.6000,  7.2917,  7.6000,  8.3333,  8.4000,  9.3750,
                 9.6000, 10.4000, 10.4167, 11.4583, 11.6000, 12.4000, 12.5000, 13.5417,
                13.6000, 14.5833, 14.6000, 15.6000, 15.6250, 16.6667, 17.7083, 18.7500,
                19.7917, 20.8333, 21.8750, 22.9167, 23.9583, 25.0000
        ], device=device)
        
        self.initialize_ee_parameters()
        


    def initialize_ee_parameters(self):
        """Initialize EE-specific parameters"""
        self.l_ee = nn.Parameter(torch.tensor(0.7823, device=device))
        self.coeffs_ee = nn.Parameter(torch.tensor([
        0.0000e+00,  2.5454e-37,  1.7567e+00,  2.5439e-01,  1.5825e+01,
                 2.0229e-01, -2.2941e-01, -7.1530e-01,  1.5236e+00,  9.7007e-02,
                -1.0322e-01, -1.1479e+00,  6.7940e-02,  2.6715e-01,  3.8370e-02,
                 8.2383e-01, -2.5215e-02, -1.1035e-02, -5.2226e-01, -9.0709e-02,
                 3.0975e-02, -2.5600e-02,  1.1581e-01, -7.9956e-02,  1.1750e-02,
                -5.3828e-02, -9.0550e-03,  4.3100e-03,  4.0257e-02,  8.6818e-02,
                -1.0210e-01, -2.0332e-02,  9.9552e-02, -1.7263e-02, -4.0245e-02,
                 1.3637e-03,  3.8439e-02, -3.1483e-02], device=device))

    def calculate_kernels(self, distances, S_points, l):
        l = torch.clamp(l, min=1e-3)  # Ensure l > 0
        # Compute squared differences for all pairs (distances, S_points)
        diff = (distances.unsqueeze(1) - S_points) ** 2 / (2 * (l ** 2))  # Ensure 3D shape
        # Compute Gaussian kernel
        kernel_values = torch.exp(-diff)
        # Compute cutoff value at r = 3l
        cutoff_value = torch.exp(-((3 * l) ** 2) / (2 * (l ** 2)))
        # Apply cutoff: subtract cutoff_value
        kernel_values = kernel_values.clone() - cutoff_value  
        # Apply masking for cutoff
        mask = torch.abs(distances.unsqueeze(1) - S_points) > 3 * l
        kernel_values = torch.where(mask, torch.tensor(0.0, device=l.device), kernel_values)
        return kernel_values

    def forward(self, precomputed):
        """Use precomputed Morse potentials and optimize only GPR components"""
        total_loss = 0.0
        
        
        ee_loss = self.process_ee_gpr(precomputed)
        total_loss += ee_loss
        
        reg = 0.01 * (
        torch.norm(self.coeffs_ee[-10:], p=1)
    )
    
        return total_loss + reg


    def process_ee_gpr(self, precomputed):
        """Optimize only EE GPR components"""
        ee_loss = 0.0
        
        for dkey in precomputed:
            # Use precomputed Morse potentials
            morse_cc = precomputed[dkey]["morse_cc"]
            morse_ec = precomputed[dkey]["morse_ec"]
            morse_ee = precomputed[dkey]["morse_ee"]
            pred_cc = precomputed[dkey]["pred_cc"]
            actual = precomputed[dkey]["actual"]
            pred_ec = precomputed[dkey]["pred_ec"]
            
            
            # Get distances for kernel calculations
            dist_ee = precomputed[dkey]["dist_ee"]  
            
            # Calculate current GPR predictions
            pred_ee_list = []
            for dist_tensor in dist_ee:
                K_ee = self.calculate_kernels(dist_tensor, self.S_points, self.l_ee)
                pred_ee = torch.einsum('ns,s->n', K_ee, self.coeffs_ee).sum()
                pred_ee_list.append(pred_ee)
                
            pred_ee_tensor = torch.stack(pred_ee_list)
            
            
            # Combine fixed CC/EC contributions, EE Morse, and trainable EE GPR
            predicted = (
                morse_cc
                + pred_cc
                + morse_ec
                + pred_ec
                + morse_ee
                + pred_ee_tensor
            )

            
            d_spacing = precomputed[dkey]["d_spacing"]

            # Energy-dependent weighting
            weights = 1.0 / (torch.abs(actual) + 1e-3)
            weights = torch.clamp(weights, min=1.0 / 90.0, max=1.0)

            # Main weighted MSE loss
            main_loss = torch.mean(weights * (predicted - actual) ** 2)

            # Additional region-specific refinement penalties
            right_threshold = 33.0 if dkey == "dataset_1" else 20.0
            transition_width = 1.0

            # Smoothly increase the penalty in the large-separation region
            right_strength = torch.sigmoid(
                (d_spacing - right_threshold) / transition_width
            )

            # Define the intermediate region
            if dkey == "dataset_1":
                middle_low, middle_high = 25.0, 33.0
            else:
                middle_low, middle_high = 10.5, 20.0

            middle_strength = (
                torch.sigmoid((d_spacing - middle_low) / transition_width)
                * torch.sigmoid((middle_high - d_spacing) / transition_width)
            )

            right_penalty = torch.mean(
                right_strength * torch.abs(predicted)
            )

            middle_penalty = 4.0 * torch.mean(
                middle_strength * (predicted - actual) ** 2
            )

            # Encourage the interaction energy to approach zero
            # beyond the selected large-separation cutoff
            cutoff = 33.0 if dkey == "dataset_1" else 20.0
            beyond_mask = d_spacing >= cutoff

            if beyond_mask.any():
                zero_target_penalty = torch.mean(predicted[beyond_mask] ** 2)
            else:
                zero_target_penalty = torch.tensor(
                    0.0,
                    device=predicted.device,
                    dtype=predicted.dtype
                )

            ee_loss += (
                main_loss
                + 0.5 * right_penalty.clamp(max=10.0)
                + 0.3 * middle_penalty.clamp(max=10.0)
                + 2.0 * zero_target_penalty
            )

        return ee_loss
    
model = CombinedGPRModel().to(device)
print("Trainable parameters:")
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.shape)
optimizer = optim.AdamW(model.parameters(), lr=0.001)
num_epochs = 100000
for epoch in range(num_epochs):
    optimizer.zero_grad()
    loss = model(precomputed)
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch}: Loss={loss.item():.6f}")
    
    if epoch % 100 == 0:

        print(f"EE l={model.l_ee:}, EE C={model.coeffs_ee:}")        
        

print(f"EE l={model.l_ee:}, EE C={model.coeffs_ee:}")                


In [ ]:
# Final EE GPR parameters used in the published model
l_ee = torch.tensor(0.7823, device=device)
coeffs_ee = torch.tensor([
        0.0000e+00,  2.5454e-37,  1.7567e+00,  2.5439e-01,  1.5825e+01,
                 2.0229e-01, -2.2941e-01, -7.1530e-01,  1.5236e+00,  9.7007e-02,
                -1.0322e-01, -1.1479e+00,  6.7940e-02,  2.6715e-01,  3.8370e-02,
                 8.2383e-01, -2.5215e-02, -1.1035e-02, -5.2226e-01, -9.0709e-02,
                 3.0975e-02, -2.5600e-02,  1.1581e-01, -7.9956e-02,  1.1750e-02,
                -5.3828e-02, -9.0550e-03,  4.3100e-03,  4.0257e-02,  8.6818e-02,
                -1.0210e-01, -2.0332e-02,  9.9552e-02, -1.7263e-02, -4.0245e-02,
                 1.3637e-03,  3.8439e-02, -3.1483e-02], device=device)
    
    


def predict_energy(data):
    """Predict energy using optimized parameters."""
    predicted_energies = {}

    for dkey in data.keys():
        dist_ee = data[dkey]["edge_edge"]  # List of tensors
        actual = data[dkey]["actual"]  # Tensor of actual energies

        morse_cc_tensor = precomputed[dkey]["morse_cc"]
        morse_ec_tensor = precomputed[dkey]["morse_ec"]
        morse_ee_tensor = precomputed[dkey]["morse_ee"]
        
        pred_cc_tensor  = precomputed[dkey]["pred_cc"]
        pred_ec_tensor  = precomputed[dkey]["pred_ec"]
        
    
            # Kernel predictions
        pred_ee_list = []
        for dist_tensor in dist_ee:
            K_ee = calculate_kernels(dist_tensor, S_points, l_ee)
            pred_ee = torch.einsum('ns,s->n', K_ee, coeffs_ee).sum()
            pred_ee_list.append(pred_ee)
                
        pred_ee_tensor = torch.stack(pred_ee_list)
        predicted = morse_cc_tensor + pred_cc_tensor + morse_ec_tensor + pred_ec_tensor + morse_ee_tensor + pred_ee_tensor
        predicted_energies[dkey] = predicted

    return predicted_energies


def plot_results(dkey, processed_data, predicted_energies, zoom=False):
    actual = processed_data[dkey]["actual"].cpu().numpy()
    predicted = predicted_energies[dkey].cpu().numpy()
    xvals = processed_data[dkey]["d_spacing_energy"].cpu().numpy()
    
    plt.figure(figsize=(8, 6))
    plt.plot(xvals, actual, label="Atomistic", alpha=0.7)
    plt.scatter(xvals, predicted,  label="Mesoscale", s=5)
    
    plt.xlabel("d-spacing (Å)", fontsize=12)
    plt.ylabel("PMF (kcal/mol)", fontsize=12)
    plt.title(f"Predicted vs Actual Energies: {dkey}", fontsize=14)
    plt.legend(fontsize=12)
    
#     if zoom:
#         plt.ylim([-2, 5])
#         plt.xlim([26, 45])
#         plt.xticks(np.arange(23, 45, 1))
#     else:
#         plt.xticks(np.arange(23, 43, 2))
#         plt.xlim([23,35])
    
    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()


    
    
predicted_energies = predict_energy(processed_data)

for dkey in processed_data.keys():
    plot_results(dkey, processed_data, predicted_energies)

